<a href="https://colab.research.google.com/github/Deepu-shrivas/DSA-PROBLEMS/blob/main/fake_news_detection2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import files
uploaded=files.upload()
print(uploaded)

Saving fake_news_dataset.csv to fake_news_dataset (1).csv
{'fake_news_dataset (1).csv': b'title,text,label\nScientists discover a new method to clean polluted water using sunlight.,Scientists have developed a solar-powered method that can help remove pollutants from water.,REAL\nGovernment launches a new digital education platform for students.,The education platform provides online learning resources and study materials.,REAL\nResearchers develop low-cost battery that charges in minutes.,A research team has demonstrated a battery design aimed at reducing charging time.,REAL\nLocal hospital opens a new emergency care unit.,The hospital has opened an expanded emergency department to provide faster medical care.,REAL\nNew public transport buses introduced to reduce city pollution.,The transport authority has added electric buses to its public transportation fleet.,REAL\nUniversity announces scholarships for engineering students.,The university has announced scholarships for eligible engi

In [4]:
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

importing the Dependencies

In [5]:
import numpy as np
import pandas as pd
import re
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [6]:
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [7]:
#printing the stopwords in English
print(stopwords.words('english'))

['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an', 'and', 'any', 'are', 'aren', "aren't", 'as', 'at', 'be', 'because', 'been', 'before', 'being', 'below', 'between', 'both', 'but', 'by', 'can', 'couldn', "couldn't", 'd', 'did', 'didn', "didn't", 'do', 'does', 'doesn', "doesn't", 'doing', 'don', "don't", 'down', 'during', 'each', 'few', 'for', 'from', 'further', 'had', 'hadn', "hadn't", 'has', 'hasn', "hasn't", 'have', 'haven', "haven't", 'having', 'he', "he'd", "he'll", 'her', 'here', 'hers', 'herself', "he's", 'him', 'himself', 'his', 'how', 'i', "i'd", 'if', "i'll", "i'm", 'in', 'into', 'is', 'isn', "isn't", 'it', "it'd", "it'll", "it's", 'its', 'itself', "i've", 'just', 'll', 'm', 'ma', 'me', 'mightn', "mightn't", 'more', 'most', 'mustn', "mustn't", 'my', 'myself', 'needn', "needn't", 'no', 'nor', 'not', 'now', 'o', 'of', 'off', 'on', 'once', 'only', 'or', 'other', 'our', 'ours', 'ourselves', 'out', 'over', 'own', 're', 's', 'same', 'shan', "shan't", 'she

Data preprocessing


In [8]:
# loading the dataset to a pandas  DataFrame
news_dataset = pd.read_csv('/content/fake_news_dataset.csv')

In [10]:
news_dataset.shape

(20, 3)

In [9]:
# print the first5 rows of the dataframe
news_dataset.head()

,title,text,label
0,Scientists discover a new method to clean poll...,Scientists have developed a solar-powered meth...,REAL
1,Government launches a new digital education pl...,The education platform provides online learnin...,REAL
2,Researchers develop low-cost battery that char...,A research team has demonstrated a battery des...,REAL
3,Local hospital opens a new emergency care unit.,The hospital has opened an expanded emergency ...,REAL
4,New public transport buses introduced to reduc...,The transport authority has added electric bus...,REAL


In [11]:
# counting thenumber of missing values in dataset
news_dataset.isnull().sum()

,0
title,0
text,0
label,0


In [12]:
#replacing null values with empty string
news_dataset=news_dataset.fillna('')

In [13]:
#merging the news title and label
news_dataset['content']=news_dataset['title']+' '+news_dataset['label']

In [14]:
print(news_dataset['content'])

0     Scientists discover a new method to clean poll...
1     Government launches a new digital education pl...
2     Researchers develop low-cost battery that char...
3     Local hospital opens a new emergency care unit...
4     New public transport buses introduced to reduc...
5     University announces scholarships for engineer...
6     Weather department issues heavy rainfall advis...
7     Researchers publish study on improving crop pr...
8     City opens a new public library for students a...
9     Scientists test a new system for detecting for...
10    BREAKING: Drinking one glass of special water ...
11    Scientists secretly confirm that humans can li...
12    Government will give every citizen one crore r...
13    Aliens land in India and government hides the ...
14    Eating only chocolate for seven days guarantee...
15    New phone app can read people's thoughts throu...
16    Drinking hot water instantly cures every type ...
17    Scientists announce that the moon will dis

In [15]:
#seprating the data & label
X = news_dataset.drop(columns='label', axis=1)
Y = news_dataset['label']

In [16]:
print(X)
print(Y)

                                                title  \
0   Scientists discover a new method to clean poll...   
1   Government launches a new digital education pl...   
2   Researchers develop low-cost battery that char...   
3     Local hospital opens a new emergency care unit.   
4   New public transport buses introduced to reduc...   
5   University announces scholarships for engineer...   
6   Weather department issues heavy rainfall advis...   
7   Researchers publish study on improving crop pr...   
8   City opens a new public library for students a...   
9   Scientists test a new system for detecting for...   
10  BREAKING: Drinking one glass of special water ...   
11  Scientists secretly confirm that humans can li...   
12  Government will give every citizen one crore r...   
13  Aliens land in India and government hides the ...   
14  Eating only chocolate for seven days guarantee...   
15  New phone app can read people's thoughts throu...   
16  Drinking hot water instantl

Stemming:

Stemming is the  process of reducing a word to its Root word
example: actor,actress,acting --> act



In [17]:
port_stem = PorterStemmer()


In [18]:
def Stemming(content):
  stemmed_content = re.sub('[^a-zA-Z]',' ',content)
  stemmed_content = stemmed_content.lower()
  stemmed_content = stemmed_content.split()
  stemmed_content = [port_stem.stem(word) for word in stemmed_content if not word in stopwords.words('english')]
  stemmed_content = ''.join(stemmed_content)
  return stemmed_content

In [19]:
news_dataset['content']=news_dataset['content'].apply(Stemming)

In [20]:
print(news_dataset['content'])

0     scientistdiscovnewmethodcleanpollutwaterusesun...
1           governlaunchnewdigiteducplatformstudentreal
2           researchdeveloplowcostbatterichargminutreal
3                   localhospitopennewemergcareunitreal
4     newpublictransportbuseintroducreduccitipollutreal
5             universannouncscholarshipenginstudentreal
6     weatherdepartissuheavirainfaladvisoriseverregi...
7     researchpublishstudiimprovcropproductsmartirri...
8              citiopennewpubliclibraristudentresidreal
9       scientisttestnewsystemdetectforestfireearlireal
10    breakdrinkoneglassspecialwatermakepermanimmune...
11    scientistsecretliconfirmhumanlivewithoutsleepfake
12       governgiveevericitizenonecrorerupenextweekfake
13                     alienlandindiagovernhideevidfake
14              eatchocolsevendayguaranteweightlossfake
15                newphoneappreadpeoplthoughtcamerafake
16        drinkhotwaterinstantlicureeveritypeinfectfake
17            scientistannouncmoondisappearthree

In [21]:
#seprating a data and label
X = news_dataset['content'].values
Y = news_dataset['label'].values

In [22]:
print(X)

['scientistdiscovnewmethodcleanpollutwaterusesunlightreal'
 'governlaunchnewdigiteducplatformstudentreal'
 'researchdeveloplowcostbatterichargminutreal'
 'localhospitopennewemergcareunitreal'
 'newpublictransportbuseintroducreduccitipollutreal'
 'universannouncscholarshipenginstudentreal'
 'weatherdepartissuheavirainfaladvisoriseverregionreal'
 'researchpublishstudiimprovcropproductsmartirrigreal'
 'citiopennewpubliclibraristudentresidreal'
 'scientisttestnewsystemdetectforestfireearlireal'
 'breakdrinkoneglassspecialwatermakepermanimmuneveridiseasfake'
 'scientistsecretliconfirmhumanlivewithoutsleepfake'
 'governgiveevericitizenonecrorerupenextweekfake'
 'alienlandindiagovernhideevidfake'
 'eatchocolsevendayguaranteweightlossfake'
 'newphoneappreadpeoplthoughtcamerafake'
 'drinkhotwaterinstantlicureeveritypeinfectfake'
 'scientistannouncmoondisappearthreedayfake'
 'secretmachinturnordinaripapergoldhomefake'
 'schoolpermancloseartificiintelligreplacteacherfake']


In [23]:
print(Y)

['REAL' 'REAL' 'REAL' 'REAL' 'REAL' 'REAL' 'REAL' 'REAL' 'REAL' 'REAL'
 'FAKE' 'FAKE' 'FAKE' 'FAKE' 'FAKE' 'FAKE' 'FAKE' 'FAKE' 'FAKE' 'FAKE']


In [24]:
# converting the textual data to numerical data
vectorizer = TfidfVectorizer()
vectorizer.fit(X)

X = vectorizer.transform(X)

In [25]:
print(X)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 20 stored elements and shape (20, 20)>
  Coords	Values
  (0, 14)	1.0
  (1, 6)	1.0
  (2, 10)	1.0
  (3, 7)	1.0
  (4, 9)	1.0
  (5, 18)	1.0
  (6, 19)	1.0
  (7, 11)	1.0
  (8, 2)	1.0
  (9, 16)	1.0
  (10, 1)	1.0
  (11, 15)	1.0
  (12, 5)	1.0
  (13, 0)	1.0
  (14, 4)	1.0
  (15, 8)	1.0
  (16, 3)	1.0
  (17, 13)	1.0
  (18, 17)	1.0
  (19, 12)	1.0


spliting a dataset to training & test data



In [26]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size = 0.2, stratify=Y, random_state=2)

training the model : Logistic regression model


In [27]:
model = LogisticRegression()

In [28]:
model.fit(X_train, Y_train)

LogisticRegression()

Evaluation


accuracy score


In [29]:
#accuracy score on the training data
X_train_prediction = model.predict(X_train)
training_data_accuracy = accuracy_score(X_train_prediction, Y_train)

In [31]:
print('Accuracy score of the training data :', training_data_accuracy)

Accuracy score of the training data : 1.0


In [30]:
#accuracy score on the training data
X_test_prediction = model.predict(X_test)
test_data_accuracy = accuracy_score(X_test_prediction, Y_test)

In [32]:
X_test_prediction = model.predict(X_test)
test_data_accuracy = accuracy_score(X_test_prediction, Y_test)
print('Accuracy score of the test data :', test_data_accuracy)

Accuracy score of the test data : 0.5


Making a prediction system

In [33]:
X_new = X_test[0]

prediction = model.predict(X_new)
print(prediction)

if (prediction[0]==0):
  print('The news is Real')
else:
  print('The news is Fake')

['FAKE']
The news is Fake


In [34]:
print(Y_test[3])

FAKE
